# 01 — Data Preparation

Downloads PubMedQA and MedQA datasets, formats them for QLoRA fine-tuning
using Gemma 4 prompt format, and runs baseline evaluation on the unmodified
Gemma 4 E4B model.

In [1]:
# Install dependencies
!pip install -q "transformers>=5.5.0" peft bitsandbytes datasets accelerate sentencepiece protobuf scipy pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 92.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.8 MB/s eta 0:00:00


In [2]:
# Login to HuggingFace (needed for Gemma access)
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## 1. Load and Explore Datasets

In [3]:
from datasets import load_dataset

# Using the standard pubmed_qa path which typically provides pre-converted Parquet files
# and avoids the 'bigbio' script-based loading error.
pubmedqa = load_dataset("pubmed_qa", "pqa_labeled")

print("PubMedQA splits:", {k: len(v) for k, v in pubmedqa.items()})
print("\nSample:")
print(pubmedqa["train"][0])

README.md: 0.00B [00:00, ?B/s]

pqa_labeled/train-00000-of-00001.parquet:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

PubMedQA splits: {'train': 1000}

Sample:
{'pubid': 21645374, 'question': 'Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?', 'context': {'contexts': ['Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants.', 'The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells

In [4]:
# MedQA — USMLE-style multiple choice
# Using a Parquet-based repository to avoid the loading script error
medqa = load_dataset("GBaker/MedQA-USMLE-4-options")

print("MedQA splits:", {k: len(v) for k, v in medqa.items()})
print("\nSample:")
medqa["train"][0]

README.md:   0%|          | 0.00/654 [00:00<?, ?B/s]

phrases_no_exclude_train.jsonl:   0%|          | 0.00/16.2M [00:00<?, ?B/s]

phrases_no_exclude_test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/10178 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1273 [00:00<?, ? examples/s]

MedQA splits: {'train': 10178, 'test': 1273}

Sample:


{'question': 'A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ago and has been worsening despite drinking more water and taking cranberry extract. She otherwise feels well and is followed by a doctor for her pregnancy. Her temperature is 97.7°F (36.5°C), blood pressure is 122/77 mmHg, pulse is 80/min, respirations are 19/min, and oxygen saturation is 98% on room air. Physical exam is notable for an absence of costovertebral angle tenderness and a gravid uterus. Which of the following is the best treatment for this patient?',
 'answer': 'Nitrofurantoin',
 'options': {'A': 'Ampicillin',
  'B': 'Ceftriaxone',
  'C': 'Doxycycline',
  'D': 'Nitrofurantoin'},
 'meta_info': 'step2&3',
 'answer_idx': 'D',
 'metamap_phrases': ['23 year old pregnant woman',
  'weeks presents',
  'burning',
  'urination',
  'states',
  'started 1 day',
  'worsening',
  'drinking',
  'water',
  'taking cranberry extract',
  'feels well',
  'follo

## 2. Format for Training

We format each example using the **Gemma 4 prompt format**:
```
<|turn>user
{question}<turn|>
<|turn>model
{answer}<turn|>
```

In [5]:
PUBMEDQA_TEMPLATE = """<|turn>user
Context: {context}

Question: {question}
Answer with yes, no, or maybe and explain your reasoning.<turn|>
<|turn>model
{answer}<turn|>"""

MEDQA_TEMPLATE = """<|turn>user
{question}<turn|>
<|turn>model
{answer}<turn|>"""

def format_pubmedqa(example):
    # Standard pubmed_qa uses lowercase keys and nested context structure
    ctx_data = example.get("context", {})
    contexts = ctx_data.get("contexts", [])
    context_str = " ".join(contexts) if isinstance(contexts, list) else str(contexts)

    long_answer = example.get("long_answer", "")
    final_decision = example.get("final_decision", "")
    answer = f"{final_decision}. {long_answer}" if long_answer else final_decision

    return {"text": PUBMEDQA_TEMPLATE.format(context=context_str, question=example["question"], answer=answer)}

def format_medqa(example):
    question = example["question"]
    options = example.get("options", {})
    if isinstance(options, dict):
        opts_str = "\n".join(f"  {k}) {v}" for k, v in sorted(options.items()))
        question = f"{question}\n{opts_str}"

    answer_idx = example.get("answer_idx", "")
    answer = example.get("answer", "")
    full_answer = f"The answer is {answer_idx}) {answer}" if answer_idx else answer
    return {"text": MEDQA_TEMPLATE.format(question=question, answer=full_answer)}

# Apply formatting
pubmedqa_fmt = pubmedqa.map(format_pubmedqa, remove_columns=pubmedqa["train"].column_names)
medqa_fmt = medqa.map(format_medqa, remove_columns=medqa["train"].column_names)

print("Formatted PubMedQA sample:")
print(pubmedqa_fmt["train"][0]["text"][:500])
print("\n---\n")
print("Formatted MedQA sample:")
print(medqa_fmt["train"][0]["text"][:500])

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10178 [00:00<?, ? examples/s]

Map:   0%|          | 0/1273 [00:00<?, ? examples/s]

Formatted PubMedQA sample:
<|turn>user
Context: Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in anima

---

Formatted MedQA sample:
<|turn>user
A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ago and has been worsening despite drinking more water and taking cranberry extract. She otherwise feels well and is followed by a doctor for her pregnancy. Her temperature is 97.7°F (36.5°C), blood pressure is 122/77 mmHg, pulse is 80/min, respirations are 19/min, and oxygen saturation is 98% on room air. Physi

In [6]:
# Combine PubMedQA + MedQA training data
from datasets import concatenate_datasets

# PubMedQA only has 'train'. MedQA has 'train' and 'test' (which acts as validation/test).
# We will use the 'train' splits for training and the MedQA 'test' split for validation.
train_dataset = concatenate_datasets([pubmedqa_fmt["train"], medqa_fmt["train"]])

# Since PubMedQA lacks a validation split, we'll use MedQA's test split as our validation set.
val_dataset = medqa_fmt["test"]

print(f"Combined train: {len(train_dataset)} examples")
print(f"Combined val:   {len(val_dataset)} examples")

# Save to disk for use in notebook 02
import os
os.makedirs("data/train", exist_ok=True)
os.makedirs("data/val", exist_ok=True)

train_dataset.save_to_disk("data/train")
val_dataset.save_to_disk("data/val")
print("\nSaved to data/train and data/val")

Combined train: 11178 examples
Combined val:   1273 examples


Saving the dataset (0/1 shards):   0%|          | 0/11178 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1273 [00:00<?, ? examples/s]


Saved to data/train and data/val


## 3. Baseline Evaluation — Gemma 4 E4B BF16

Run the base (non-fine-tuned) model on our benchmarks to establish the control numbers

In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "google/gemma-4-E4B"

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print(f"Model loaded. Device: {model.device}")
print(f"Parameters: {model.num_parameters() / 1e9:.1f}B")

Loading google/gemma-4-E4B...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/881 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Model loaded. Device: cuda:0
Parameters: 7.9B


In [8]:
import time
import pandas as pd
from datasets import load_dataset
from tqdm import tqdm
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
results = {"model": "gemma4-e4b-bf16-baseline"}
BATCH_SIZE = 2

# --- Perplexity: WikiText-2 ---
print("[1/4] WikiText-2 perplexity...")
wiki = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
wiki_texts = [t for t in wiki["text"] if len(t.strip()) > 50][:512]

model.eval()
total_loss, total_tokens = 0.0, 0
for i in tqdm(range(0, len(wiki_texts), BATCH_SIZE)):
    batch = wiki_texts[i:i+BATCH_SIZE]
    enc = tokenizer(batch, return_tensors="pt", truncation=True, max_length=512, padding=True).to(device)
    with torch.no_grad():
        out = model(**enc, labels=enc["input_ids"])
    n = enc["attention_mask"].sum().item()
    total_loss += out.loss.item() * n
    total_tokens += n

results["perplexity_wikitext"] = torch.exp(torch.tensor(total_loss / total_tokens)).item()
print(f"  -> {results['perplexity_wikitext']:.2f}")

[1/4] WikiText-2 perplexity...


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

  0%|          | 1/256 [00:29<2:06:27, 29.76s/it]


OutOfMemoryError: CUDA out of memory. Tried to allocate 5.25 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.41 GiB is free. Including non-PyTorch memory, this process has 13.15 GiB memory in use. Of the allocated memory 8.19 GiB is allocated by PyTorch, and 4.84 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# --- Perplexity: Medical text ---
print("[2/4] Medical text perplexity...")
pqa = load_dataset("bigbio/pubmed_qa", "pubmed_qa_labeled_fold0_source", split="test")
med_texts = []
for ex in pqa:
    ctx = " ".join(ex["CONTEXTS"]) if isinstance(ex["CONTEXTS"], list) else ex["CONTEXTS"]
    if len(ctx.strip()) > 50:
        med_texts.append(ctx)
    if len(med_texts) >= 512:
        break

total_loss, total_tokens = 0.0, 0
for i in tqdm(range(0, len(med_texts), 4)):
    batch = med_texts[i:i+4]
    enc = tokenizer(batch, return_tensors="pt", truncation=True, max_length=512, padding=True).to(device)
    with torch.no_grad():
        out = model(**enc, labels=enc["input_ids"])
    n = enc["attention_mask"].sum().item()
    total_loss += out.loss.item() * n
    total_tokens += n

results["perplexity_medical"] = torch.exp(torch.tensor(total_loss / total_tokens)).item()
print(f"  -> {results['perplexity_medical']:.2f}")

In [ ]:
# --- PubMedQA accuracy ---
print("[3/4] PubMedQA accuracy...")
correct, total = 0, 0
MAX_SAMPLES = 500

for ex in tqdm(pqa, total=min(len(pqa), MAX_SAMPLES)):
    if total >= MAX_SAMPLES:
        break
    context = " ".join(ex["CONTEXTS"]) if isinstance(ex["CONTEXTS"], list) else ex["CONTEXTS"]
    prompt = (
        f"<|turn>user\n"
        f"Context: {context}\n\n"
        f"Question: {ex['QUESTION']}\n"
        f"Answer with exactly one word: yes, no, or maybe.<turn|>\n"
        f"<|turn>model\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=10, do_sample=False)
    resp = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip().lower()
    pred = resp.split()[0].strip(".,!?;:") if resp.split() else ""
    gold = ex["final_decision"].lower().strip()
    if pred in ("yes", "no", "maybe") and pred == gold:
        correct += 1
    total += 1

results["pubmedqa_accuracy"] = correct / total if total else 0
print(f"  -> {results['pubmedqa_accuracy']:.4f} ({correct}/{total})")

In [ ]:
# --- Inference speed & memory ---
print("[4/4] Inference speed & memory...")

# Speed
prompt = "<|turn>user\nWhat is diabetes?<turn|>\n<|turn>model\n"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

# Warmup
with torch.no_grad():
    model.generate(**inputs, max_new_tokens=50, do_sample=False)

total_tok, total_t = 0, 0.0
for _ in range(20):
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    torch.cuda.synchronize()
    total_tok += out.shape[1] - inputs["input_ids"].shape[1]
    total_t += time.perf_counter() - t0

results["tokens_per_sec"] = total_tok / total_t
results["peak_vram_gb"] = torch.cuda.max_memory_allocated() / (1024**3)
print(f"  -> {results['tokens_per_sec']:.1f} tok/s")
print(f"  -> {results['peak_vram_gb']:.2f} GB peak VRAM")

In [ ]:
# Save baseline results
import os
os.makedirs("results/tables", exist_ok=True)

df = pd.DataFrame([results])
df.to_csv("results/tables/all_results.csv", index=False)
print("\nBaseline results:")
df

In [ ]:
# Cleanup
del model
torch.cuda.empty_cache()
print("Done! Proceed to notebook 02 for QLoRA fine-tuning.")